In [55]:
# Imports
import os
import re
import time
import requests
import numpy as np
import rasterio
from rasterio.merge import merge
import matplotlib.pyplot as plt
import geopandas as gpd
from rasterio.mask import mask
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
import h5py
from datetime import datetime
from requests.auth import HTTPBasicAuth

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configuração dinâmica

In [26]:
# Pasta no Google Drive (resultado final)
drive_output_dir = "/content/drive/MyDrive/viirs_output"
os.makedirs(drive_output_dir, exist_ok=True)

# Pasta temporária local (mais rápido)
temp_dir = "/content/temp_viirs"
os.makedirs(temp_dir, exist_ok=True)

print("Drive:", drive_output_dir)
print("Temp:", temp_dir)

DATA_DIR = temp_dir
OUT_DIR = drive_output_dir

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Usuario e senha da NASA.
auth = HTTPBasicAuth("USUARIO", "SENHA")
DATE = "2025-04-10"
TILE = "h13v10"

# AOI
lon_min, lon_max = -53.5, -45
lat_min, lat_max = -19.5, -11.8

# Folder do shapefile no drive
SHAPEFILE = "/content/drive/MyDrive/GO/GO_UF_2024.shp"


Drive: /content/drive/MyDrive/viirs_output
Temp: /content/temp_viirs


In [4]:

def get_doy(date):
    return datetime.strptime(date, "%Y-%m-%d").strftime("%j")


In [47]:
class SessionWithHeaderRedirection(requests.Session):
    AUTH_HOST = 'urs.earthdata.nasa.gov'

    def __init__(self, auth):
        super().__init__()
        self.auth = auth

    def rebuild_auth(self, prepared_request, response):
        """
        Mantém auth mesmo após redirect entre hosts
        """
        headers = prepared_request.headers
        url = prepared_request.url

        if 'Authorization' in headers:
            original_host = requests.utils.urlparse(response.request.url).hostname
            redirect_host = requests.utils.urlparse(url).hostname

            if (original_host != redirect_host) and (redirect_host != self.AUTH_HOST):
                del headers['Authorization']

In [48]:
# Usa sessão para poder baixar dados mesmo tendo autenticação
def get_session():

    session = SessionWithHeaderRedirection(auth)
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    return session

## Download com seleção automática

In [60]:
def safe_download(session, url, out, max_retries=3):

    for attempt in range(max_retries):

        try:
            print(f"Tentativa {attempt+1}:", url.split("/")[-1])

            r = session.get(url, allow_redirects=True, timeout=60)

            if r.status_code != 200:
                raise Exception(f"HTTP {r.status_code}")

            content_type = r.headers.get("Content-Type", "")

            if "text/html" in content_type:
                raise Exception("Recebeu HTML (auth falhou no redirect)")

            with open(out, "wb") as f:
                f.write(r.content)

            size = os.path.getsize(out)

            return out

        except Exception as e:

            print("Erro:", e)

            # remove arquivo corrompido
            if os.path.exists(out):
                os.remove(out)

            if attempt < max_retries - 1:
                print("Retrying...\n")
                time.sleep(2)
            else:
                raise Exception(f"Falha após {max_retries} tentativas: {url}")

In [51]:
def download_viirs(product):

    session = get_session()

    year = DATE[:4]
    doy = get_doy(DATE)

    base_url = f"https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/{product}/{year}/{doy}"
    json_url = base_url + ".json"

    print("Acessando:", json_url)

    r = session.get(json_url)

    if r.status_code != 200:
        raise Exception(f"Erro HTTP {r.status_code}")

    data = r.json()
    files = data.get("content", [])

    if not files:
        raise Exception(f"Nenhum dado disponível para {DATE}")

    # apenas HDF5
    hdfs = [f for f in files if f["name"].endswith(".h5")]

    print(f"{len(hdfs)} tiles encontrados")

    # filtro espacial
    hdfs = filter_tiles_by_bbox(hdfs, lon_min, lon_max, lat_min, lat_max)

    if not hdfs:
        print("Nenhum tile no bbox → usando todos")
        hdfs = [f for f in files if f["name"].endswith(".h5")]

    print(f"{len(hdfs)} tiles após filtro")

    paths = []

    for f in hdfs:

        name = f["name"]
        file_url = f["downloadsLink"]

        out = os.path.join(DATA_DIR, name)

        # valida arquivo existente
        def is_valid(file):
            return os.path.exists(file) and os.path.getsize(file) > 1_000_000

        if not is_valid(out):

            print("Downloading:", name)

            # 🔥 AQUI ESTÁ A CORREÇÃO REAL
            safe_download(session, file_url, out)

        else:
            print("Usando cache:", name)

        paths.append(out)

    return paths

In [28]:
def filter_tiles_by_bbox(files, lon_min, lon_max, lat_min, lat_max):

    # aproximação simples de tiles relevantes
    valid = []

    for f in files:
        name = f["name"]

        # extrai hXXvYY
        import re
        match = re.search(r"h(\d{2})v(\d{2})", name)

        if not match:
            continue

        h = int(match.group(1))
        v = int(match.group(2))

        # regra aproximada para América do Sul
        if 10 <= h <= 14 and 9 <= v <= 12:
            valid.append(f)

    return valid

## Buscar banda automaticamente

In [6]:

def find_dataset(hdf, keyword):

    with h5py.File(hdf, "r") as f:
        matches = []

        def walk(name, obj):
            if isinstance(obj, h5py.Dataset) and keyword in name:
                matches.append(name)

        f.visititems(walk)

    if not matches:
        raise Exception(f"{keyword} não encontrado")

    return matches[0]


## Extração + reprojeção real

In [31]:
def extract_reproject(hdf, dataset, out_path):

    with h5py.File(hdf, "r") as f:
        data = f[dataset][:].astype("float32")

    # Criar raster inicial (sem georef)
    temp = out_path.replace(".tif", "_raw.tif")

    height, width = data.shape

    transform = from_bounds(-180, -90, 180, 90, width, height)

    with rasterio.open(
        temp,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="float32",
        crs="EPSG:4326",
        transform=transform
    ) as dst:
        dst.write(data, 1)

    # Reprojetar (mantém EPSG:4326 mas força consistência)
    with rasterio.open(temp) as src:

        transform, width, height = calculate_default_transform(
            src.crs, "EPSG:4326", src.width, src.height, *src.bounds
        )

        kwargs = src.meta.copy()
        kwargs.update({
            "crs": "EPSG:4326",
            "transform": transform,
            "width": width,
            "height": height
        })

        with rasterio.open(out_path, "w", **kwargs) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs="EPSG:4326",
                resampling=Resampling.nearest
            )

    os.remove(temp)

    return out_path


In [32]:
def process_tiles(files, band_name, prefix):

    outputs = []

    for i, f in enumerate(files):

        print(f"Processando {band_name} | tile {i+1}/{len(files)}")

        dataset = find_dataset(f, band_name)

        out_path = os.path.join(DATA_DIR, f"{prefix}_{i}.tif")

        tif = extract_reproject(f, dataset, out_path)
        tif = crop(tif)

        outputs.append(tif)

    return outputs

In [33]:
def mosaic(tifs, out_path):

    srcs = [rasterio.open(t) for t in tifs]

    mosaic_arr, transform = merge(srcs)

    meta = srcs[0].meta.copy()
    meta.update({
        "height": mosaic_arr.shape[1],
        "width": mosaic_arr.shape[2],
        "transform": transform
    })

    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(mosaic_arr)

    for s in srcs:
        s.close()

    return out_path

## Crop

In [8]:

def crop(tif):

    gdf = gpd.read_file(SHAPEFILE)

    with rasterio.open(tif) as src:
        gdf = gdf.to_crs(src.crs)
        out, transform = mask(src, gdf.geometry, crop=True)

        meta = src.meta.copy()
        meta.update({
            "height": out.shape[1],
            "width": out.shape[2],
            "transform": transform
        })

    out_tif = tif.replace(".tif", "_clip.tif")

    with rasterio.open(out_tif, "w", **meta) as dst:
        dst.write(out)

    return out_tif


In [9]:

def read(tif):
    with rasterio.open(tif) as src:
        return src.read(1)


In [10]:

def preview(arr, name, cmap="turbo", vmin=None, vmax=None):

    plt.figure(figsize=(8,6))
    im = plt.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im)

    plt.title(name)
    plt.axis("off")

    plt.savefig(os.path.join(OUT_DIR, name + ".png"), dpi=150, bbox_inches="tight")
    plt.close()


## Pipeline completo

In [61]:

# DOWNLOAD
sr_files = download_viirs("VNP09GA")
th_files = download_viirs("VNP21A1D")

# PROCESSAMENTO POR TILE POR BANDA
m5_tiles = process_tiles(sr_files, "M5", "m5")
m11_tiles = process_tiles(sr_files, "M11", "m11")

m13_tiles = process_tiles(th_files, "Emis_14", "m13")
m15_tiles = process_tiles(th_files, "LST_1KM", "m15")
m16_tiles = process_tiles(th_files, "Emis_15", "m16")

# MOSAICO FINAL
m5_path = mosaic(m5_tiles, os.path.join(DATA_DIR, "m5_mosaic.tif"))
m11_path = mosaic(m11_tiles, os.path.join(DATA_DIR, "m11_mosaic.tif"))

m13_path = mosaic(m13_tiles, os.path.join(DATA_DIR, "m13_mosaic.tif"))
m15_path = mosaic(m15_tiles, os.path.join(DATA_DIR, "m15_mosaic.tif"))
m16_path = mosaic(m16_tiles, os.path.join(DATA_DIR, "m16_mosaic.tif"))

Acessando: https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/VNP09GA/2025/100.json
460 tiles encontrados
20 tiles após filtro
Usando cache: VNP09GA.A2025100.h10v09.002.2025101100046.h5
Usando cache: VNP09GA.A2025100.h10v10.002.2025101100342.h5
Usando cache: VNP09GA.A2025100.h10v11.002.2025101100306.h5
Usando cache: VNP09GA.A2025100.h10v12.002.2025101103840.h5
Usando cache: VNP09GA.A2025100.h11v09.002.2025101100229.h5
Usando cache: VNP09GA.A2025100.h11v10.002.2025101100549.h5
Usando cache: VNP09GA.A2025100.h11v11.002.2025101100434.h5
Usando cache: VNP09GA.A2025100.h11v12.002.2025101100819.h5
Usando cache: VNP09GA.A2025100.h12v09.002.2025101100147.h5
Usando cache: VNP09GA.A2025100.h12v10.002.2025101100643.h5
Usando cache: VNP09GA.A2025100.h12v11.002.2025101100459.h5
Usando cache: VNP09GA.A2025100.h12v12.002.2025101100651.h5
Usando cache: VNP09GA.A2025100.h13v09.002.2025101100113.h5
Usando cache: VNP09GA.A2025100.h13v10.002.2025101100336.h5
Usando cache: VNP09GA.A2025100.h13v11.

In [62]:

# Cálculo dos indices
m5 = read(m5_path) / 10000
m11 = read(m11_path) / 10000
m13 = read(m13_path)
m15 = read(m15_path)
m16 = read(m16_path)

ndi = (m5 - m11) / (m5 + m11 + 1e-6)
anom = np.clip(m13 - m15, -10, 50)
lst = m15 + 0.5*(m15 - m16)

# Faz o preview.png de cada indice
preview(ndi, "NDI", "RdYlBu_r", -1, 1)
preview(anom, "Anomalia", "RdYlBu_r", -10, 50)
preview(lst, "LST", "Oranges", 0, 50)

print("Finalizado:", OUT_DIR)


Finalizado: /content/drive/MyDrive/viirs_output
